Apriori Algorithmn

In [9]:
from collections import defaultdict

def min_supp_count(min_supp, total_trans):
    count = min_supp * total_trans  # min_supp is percentage (e.g., 0.05 for 5%)
    return count

def get_frequent_itemsets(data, min_supp_count_value):

    symptom_counts = defaultdict(int)

    for i in data: #loop throgh the list and count
        for j in i:
            symptom_counts[j] += 1

    # Filter by minimum support
    frequent = {}
    for i in symptom_counts:
        count = symptom_counts[i]
        if count >= min_supp_count_value:
            frequent[frozenset([i])] = count


    list_for_rule_generation = frequent.copy() #make a copy to store all frequent items
    templist = frequent
    k = 2

    print(f"Found {len(frequent)} frequent 1 itemsets")

    #Generate larger itemsets
    while templist:
        print(f"Generating {k} itemsets")

        # Create candidate itemsets
        candidates = set()
        itemsets_list = list(templist.keys())

        # Combine itemsets
        for i in range(len(itemsets_list)):
            for j in range(i + 1, len(itemsets_list)):
                itemset1 = itemsets_list[i]
                itemset2 = itemsets_list[j]

                # Check if 2 itemsets have something in common then fuse them
                union_set = itemset1.union(itemset2)
                if len(union_set) == k:
                    candidates.add(union_set)

        if not candidates: #if nothing to combine then exits
            break

        # Count support
        candidate_support = {}
        for candidate in candidates:
            count = 0
            for transaction in data:
                if candidate.issubset(set(transaction)):
                    count += 1

            if count >= min_supp_count_value:
                candidate_support[candidate] = count

        if not candidate_support:
            break

        print(f"Found {len(candidate_support)} frequent {k}-itemsets")
        list_for_rule_generation.update(candidate_support)
        templist = candidate_support
        k += 1

    return list_for_rule_generation

def generate_association_rules(frequent_itemsets, min_confidence, total_transactions):

    rules = []

    for itemset, support_count in frequent_itemsets.items():
        if len(itemset) >= 2:  # generate rules for itemsets with at least 2 items
            items_list = list(itemset)

            # Generate all possible rules for this itemset
            # For itemset {A,B,C}, generate rules eg a-bc, b-ac ....
            from itertools import combinations

            # Generate all non-empty proper subsets on the left
            for r in range(1, len(items_list)):
                for antecedent_items in combinations(items_list, r):
                    antecedent_set = frozenset(antecedent_items)
                    consequent_set = itemset - antecedent_set

                    # Calculate confidence
                    if antecedent_set in frequent_itemsets:
                        antecedent_support = frequent_itemsets[antecedent_set] #(total set count)/(left set count)
                        confidence = support_count / antecedent_support

                        if confidence >= min_confidence: #add into rules if it is >= to the minconfidence
                            rules.append({
                                'rule_id': len(rules) + 1,
                                'antecedent': antecedent_set, #left
                                'consequent': consequent_set, #right
                                'support': support_count / total_transactions,
                                'support_count': support_count,
                                'confidence': confidence
                            })

    return rules

def apriori2(data, min_supp, confidence):

    total_count = len(data)
    min_supp_count_value = min_supp_count(min_supp, total_count)

    print(f"Apriori")
    print(f"Total transactions: {total_count}")
    print(f"Minimum support: {min_supp:.1%} (≥{min_supp_count_value})")
    print(f"Minimum confidence: {confidence:.1%}")

    frequent_itemsets = get_frequent_itemsets(data, min_supp_count_value)

    itemset_sizes = {}
    for itemset in frequent_itemsets.keys():
        size = len(itemset)
        itemset_sizes[size] = itemset_sizes.get(size, 0) + 1

    for size, count in sorted(itemset_sizes.items()):
        print(f"  {size}-itemsets: {count}")

    print(f"\nSTEP 2: GENERATING ASSOCIATION RULES...")
    rules = generate_association_rules(frequent_itemsets, confidence, total_count)

    print(f"Generated {len(rules)} association rules")

    return frequent_itemsets, rules

Import Dataset

In [10]:
import os
import pandas as pd

import pandas as pd
import os

# Load the dataset.csv file
df = pd.read_csv(r'C:\Users\rqpua\Downloads\dataset.csv')
df.head()

,Disease,Symptom_1,Symptom_2,Symptom_3,Symptom_4,Symptom_5,Symptom_6,Symptom_7,Symptom_8,Symptom_9,Symptom_10,Symptom_11,Symptom_12,Symptom_13,Symptom_14,Symptom_15,Symptom_16,Symptom_17
0,Fungal infection,itching,skin_rash,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Fungal infection,skin_rash,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Fungal infection,itching,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Fungal infection,itching,skin_rash,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Fungal infection,itching,skin_rash,nodal_skin_eruptions,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
df.head()

,Disease,Symptom_1,Symptom_2,Symptom_3,Symptom_4,Symptom_5,Symptom_6,Symptom_7,Symptom_8,Symptom_9,Symptom_10,Symptom_11,Symptom_12,Symptom_13,Symptom_14,Symptom_15,Symptom_16,Symptom_17
0,Fungal infection,itching,skin_rash,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Fungal infection,skin_rash,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Fungal infection,itching,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Fungal infection,itching,skin_rash,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Fungal infection,itching,skin_rash,nodal_skin_eruptions,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
## Data Cleaning step

df.fillna(0,inplace=True)

In [13]:
df.head()

,Disease,Symptom_1,Symptom_2,Symptom_3,Symptom_4,Symptom_5,Symptom_6,Symptom_7,Symptom_8,Symptom_9,Symptom_10,Symptom_11,Symptom_12,Symptom_13,Symptom_14,Symptom_15,Symptom_16,Symptom_17
0,Fungal infection,itching,skin_rash,nodal_skin_eruptions,dischromic _patches,0,0,0,0,0,0,0,0,0,0,0,0,0
1,Fungal infection,skin_rash,nodal_skin_eruptions,dischromic _patches,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,Fungal infection,itching,nodal_skin_eruptions,dischromic _patches,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,Fungal infection,itching,skin_rash,dischromic _patches,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,Fungal infection,itching,skin_rash,nodal_skin_eruptions,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [14]:
#See all the symptoms
symptom_columns = [col for col in df.columns if col.startswith('Symptom_')]

#Get unique ones
all_symptoms = df[symptom_columns].values.flatten()
unique_symptoms = pd.unique(all_symptoms)

# Remove any NaN values
unique_symptoms = unique_symptoms[~pd.isna(unique_symptoms)]

print(f"Found {len(unique_symptoms)} unique symptoms:")
print(unique_symptoms)

Found 132 unique symptoms:
['itching' ' skin_rash' ' nodal_skin_eruptions' ' dischromic _patches' 0
 ' continuous_sneezing' ' shivering' ' chills' ' watering_from_eyes'
 ' stomach_pain' ' acidity' ' ulcers_on_tongue' ' vomiting' ' cough'
 ' chest_pain' ' yellowish_skin' ' nausea' ' loss_of_appetite'
 ' abdominal_pain' ' yellowing_of_eyes' ' burning_micturition'
 ' spotting_ urination' ' passage_of_gases' ' internal_itching'
 ' indigestion' ' muscle_wasting' ' patches_in_throat' ' high_fever'
 ' extra_marital_contacts' ' fatigue' ' weight_loss' ' restlessness'
 ' lethargy' ' irregular_sugar_level' ' blurred_and_distorted_vision'
 ' obesity' ' excessive_hunger' ' increased_appetite' ' polyuria'
 ' sunken_eyes' ' dehydration' ' diarrhoea' ' breathlessness'
 ' family_history' ' mucoid_sputum' ' headache' ' dizziness'
 ' loss_of_balance' ' lack_of_concentration' ' stiff_neck' ' depression'
 ' irritability' ' visual_disturbances' ' back_pain' ' weakness_in_limbs'
 ' neck_pain' ' weakness_of_

In [15]:
# Clean the data 
df_clean = df.copy()
symptom_cols = [col for col in df.columns if col.startswith('Symptom_')]

#Clean the symptom strings 
for col in symptom_cols:
    df_clean[col] = df_clean[col].apply(lambda x: x.strip() if isinstance(x, str) else x)

#Apply standardization for names of similiar meaning
symptom_mapping = {
    'abdominal_pain': 'stomach_pain',
    'belly_pain': 'stomach_pain'
}

for col in symptom_cols:
    df_clean[col] = df_clean[col].replace(symptom_mapping)

#Get unique symptoms and clean them
all_symptoms_clean = df_clean[symptom_cols].values.flatten()
unique_symptoms_clean = pd.unique(all_symptoms_clean)

# Filter out NaN, 0, and empty values, then convert to strings
unique_symptoms_clean = [str(s).strip() for s in unique_symptoms_clean 
                        if not pd.isna(s) and s != 0 and s != '0' and str(s).strip() != '']

# Remove duplicates after cleaning
unique_symptoms_clean = list(set(unique_symptoms_clean))

print(f"After proper standardization: {len(unique_symptoms_clean)} unique symptoms")
print("Standardized symptoms:")
for symptom in sorted(unique_symptoms_clean):
    print(f"- {symptom}")

After proper standardization: 129 unique symptoms
Standardized symptoms:
- abnormal_menstruation
- acidity
- acute_liver_failure
- altered_sensorium
- anxiety
- back_pain
- blackheads
- bladder_discomfort
- blister
- blood_in_sputum
- bloody_stool
- blurred_and_distorted_vision
- breathlessness
- brittle_nails
- bruising
- burning_micturition
- chest_pain
- chills
- cold_hands_and_feets
- coma
- congestion
- constipation
- continuous_feel_of_urine
- continuous_sneezing
- cough
- cramps
- dark_urine
- dehydration
- depression
- diarrhoea
- dischromic _patches
- distention_of_abdomen
- dizziness
- drying_and_tingling_lips
- enlarged_thyroid
- excessive_hunger
- extra_marital_contacts
- family_history
- fast_heart_rate
- fatigue
- fluid_overload
- foul_smell_of urine
- headache
- high_fever
- hip_joint_pain
- history_of_alcohol_consumption
- increased_appetite
- indigestion
- inflammatory_nails
- internal_itching
- irregular_sugar_level
- irritability
- irritation_in_anus
- itching
- join

In [16]:
df_clean.head(4920)

,Disease,Symptom_1,Symptom_2,Symptom_3,Symptom_4,Symptom_5,Symptom_6,Symptom_7,Symptom_8,Symptom_9,Symptom_10,Symptom_11,Symptom_12,Symptom_13,Symptom_14,Symptom_15,Symptom_16,Symptom_17
0,Fungal infection,itching,skin_rash,nodal_skin_eruptions,dischromic _patches,0,0,0,0,0,0,0,0,0,0,0,0,0
1,Fungal infection,skin_rash,nodal_skin_eruptions,dischromic _patches,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,Fungal infection,itching,nodal_skin_eruptions,dischromic _patches,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,Fungal infection,itching,skin_rash,dischromic _patches,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,Fungal infection,itching,skin_rash,nodal_skin_eruptions,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4915,(vertigo) Paroymsal Positional Vertigo,vomiting,headache,nausea,spinning_movements,loss_of_balance,unsteadiness,0,0,0,0,0,0,0,0,0,0,0
4916,Acne,skin_rash,pus_filled_pimples,blackheads,scurring,0,0,0,0,0,0,0,0,0,0,0,0,0
4917,Urinary tract infection,burning_micturition,bladder_discomfort,foul_smell_of urine,continuous_feel_of_urine,0,0,0,0,0,0,0,0,0,0,0,0,0
4918,Psoriasis,skin_rash,joint_pain,skin_peeling,silver_like_dusting,small_dents_in_nails,inflammatory_nails,0,0,0,0,0,0,0,0,0,0,0


In [17]:
# Make each row as a basket and remove disease 
symptom_cols = [col for col in df_clean.columns if col.startswith('Symptom_')]

transactions_clean = []
for i in range(len(df_clean)):
    transaction = []
    for col in symptom_cols:
        symptom_value = df_clean.loc[i, col]
        if str(symptom_value) != '0' and not pd.isna(symptom_value) and str(symptom_value).strip() != '':
            transaction.append(str(symptom_value).strip())
    transactions_clean.append(transaction)

print(f"Created {len(transactions_clean)} cleaned transactions")
print("First 5 cleaned transactions:")
for i in range(5):
    print(f"{df_clean.loc[i, 'Disease']}: {transactions_clean[i]}")

Created 4920 cleaned transactions
First 5 cleaned transactions:
Fungal infection: ['itching', 'skin_rash', 'nodal_skin_eruptions', 'dischromic _patches']
Fungal infection: ['skin_rash', 'nodal_skin_eruptions', 'dischromic _patches']
Fungal infection: ['itching', 'nodal_skin_eruptions', 'dischromic _patches']
Fungal infection: ['itching', 'skin_rash', 'dischromic _patches']
Fungal infection: ['itching', 'skin_rash', 'nodal_skin_eruptions']


In [18]:
# Run Apriori with your transactions
min_support = 0.1  # 10% minimum support
min_confidence = 0.7  # 70% minimum confidence

frequent_itemsets, association_rules = apriori2(transactions_clean, min_support, min_confidence)


Apriori
Total transactions: 4920
Minimum support: 10.0% (≥492.0)
Minimum confidence: 70.0%
Found 19 frequent 1 itemsets
Generating 2 itemsets
Found 33 frequent 2-itemsets
Generating 3 itemsets
Found 19 frequent 3-itemsets
Generating 4 itemsets
Found 2 frequent 4-itemsets
Generating 5 itemsets
  1-itemsets: 19
  2-itemsets: 33
  3-itemsets: 19
  4-itemsets: 2

STEP 2: GENERATING ASSOCIATION RULES...
Generated 70 association rules


In [24]:
# Display only 2-itemsets or greater
print("🔍 ALL ITEMSETS:")
print("=" * 80)
print(f"{'2-Itemset':<50} {'Support Count':<15} {'Support %':<10}")
print("-" * 80)

total_transactions = len(transactions_clean)

# Only show itemsets > 2
two_itemsets = {itemset: count for itemset, count in frequent_itemsets.items() if len(itemset) >= 2}
two_itemsets_sorted = sorted(two_itemsets.items(), key=lambda x: x[1], reverse=True)

for itemset, support_count in two_itemsets_sorted:
    support_percentage = (support_count / total_transactions) * 100
    itemset_str = str(tuple(itemset))
    print(f"{itemset_str:<50} {support_count:<15} {support_percentage:.1f}%")

print(f"\nTotal itemsets found: {len(two_itemsets)}")

🔍 ALL ITEMSETS:
2-Itemset                                          Support Count   Support % 
--------------------------------------------------------------------------------
('nausea', 'vomiting')                             978             19.9%
('vomiting', 'stomach_pain')                       978             19.9%
('fatigue', 'high_fever')                          978             19.9%
('loss_of_appetite', 'yellowing_of_eyes')          786             16.0%
('loss_of_appetite', 'fatigue')                    774             15.7%
('loss_of_appetite', 'vomiting')                   768             15.6%
('yellowish_skin', 'stomach_pain')                 762             15.5%
('fatigue', 'vomiting')                            762             15.5%
('loss_of_appetite', 'nausea')                     666             13.5%
('malaise', 'fatigue')                             666             13.5%
('fatigue', 'nausea')                              660             13.4%
('chills', 'high_fever

In [23]:
# Analyze which diseases contain ALL itemsets (2-itemsets and larger)
print("DISEASES CONTAINING APRIORI ITEMSETS (2+ items)")
print("=" * 80)

# Define the disease column name for your dataset
disease_col = 'Disease'

# Get all itemsets with 2 or more items
multi_itemsets = {itemset: count for itemset, count in frequent_itemsets.items() if len(itemset) >= 2}
multi_itemsets_sorted = sorted(multi_itemsets.items(), key=lambda x: x[1], reverse=True)

# Precompute disease-symptom mappings for faster lookup
print("\nPrecomputing disease-symptom mappings...")
disease_symptoms = {}

for disease in df_clean[disease_col].unique():
    disease_data = df_clean[df_clean[disease_col] == disease]
    # Get all unique symptoms for this disease across all patients
    all_symptoms = set()
    for idx, patient in disease_data.iterrows():
        for col in [f'Symptom_{i}' for i in range(1, 18)]:
            if pd.notna(patient[col]) and patient[col] != '':
                all_symptoms.add(patient[col])
    disease_symptoms[disease] = all_symptoms

print("Disease-symptom mappings computed!")

itemset_to_diseases = {}
processed = 0

print(f"\nAnalyzing itemsets...")
for itemset, support_count in multi_itemsets_sorted:
    symptoms = set(itemset)  # Use set for faster operations
    processed += 1
    
    # Show progress
    if processed % 10 == 0:
        print(f"  Processed {processed}/{len(multi_itemsets_sorted)} itemsets...")
    
    # Find diseases that have ALL symptoms in this itemset
    matching_diseases = []
    
    for disease, disease_symptom_set in disease_symptoms.items():
        # Check if disease has ALL symptoms from the itemset
        if symptoms.issubset(disease_symptom_set):
            matching_diseases.append(disease)
    
    itemset_to_diseases[itemset] = {
        'diseases': matching_diseases,
        'support_count': support_count,
        'support_pct': (support_count / total_transactions) * 100
    }

print("Analysis complete!")

# Display ALL results organized by itemset size
print(f"\nCOMPLETE RESULTS - ALL ITEMSETS (2+ SYMPTOMS):")
print("=" * 70)

# Get all sizes present and sort them
all_sizes_present = sorted(set(len(itemset) for itemset in itemset_to_diseases.keys()))
print(f"Itemset sizes found: {all_sizes_present}")

for size in all_sizes_present:
    size_itemsets = [(itemset, data) for itemset, data in itemset_to_diseases.items() 
                     if len(itemset) == size]
    
    print(f"\n{'='*60}")
    print(f"{size}-SYMPTOM COMBINATIONS ({len(size_itemsets)} patterns)")
    print(f"{'='*60}")
    
    itemsets_with_diseases = 0
    
    for itemset, data in sorted(size_itemsets, key=lambda x: x[1]['support_count'], reverse=True):
        diseases_list = data['diseases']
        print(f"\n🔹 {tuple(itemset)}")
        print(f"   Support: {data['support_count']} patients ({data['support_pct']:.1f}%)")
        if diseases_list:
            itemsets_with_diseases += 1
            print(f"Associated Diseases: {', '.join(sorted(diseases_list))}")
        else:
            print(f"No disease contains all these symptoms together")
    
    print(f"\nSummary: {itemsets_with_diseases}/{len(size_itemsets)} {size}-item patterns have disease associations")

# Summary statistics
print(f"\n{'='*50}")
print(f"OVERALL SUMMARY")
print(f"{'='*50}")
total_itemsets_with_diseases = sum(1 for data in itemset_to_diseases.values() if data['diseases'])
print(f"Total itemsets analyzed: {len(multi_itemsets)}")
print(f"Itemsets with disease associations: {total_itemsets_with_diseases} ({total_itemsets_with_diseases/len(multi_itemsets)*100:.1f}%)")

# Find most common diseases across itemsets
disease_frequency = {}
for data in itemset_to_diseases.values():
    for disease in data['diseases']:
        disease_frequency[disease] = disease_frequency.get(disease, 0) + 1

if disease_frequency:
    print(f"\nTOP DISEASES BY ITEMSET ASSOCIATIONS:")
    for disease, freq in sorted(disease_frequency.items(), key=lambda x: x[1], reverse=True)[:15]:
        print(f"  {disease}: {freq} itemsets")

# Show key highlights
print(f"\n{'='*50}")
print(f"KEY HIGHLIGHTS")
print(f"{'='*50}")

# Top 3 itemsets by support
print(f"\nTOP 3 MOST FREQUENT SYMPTOM COMBINATIONS:")
top_support = sorted(itemset_to_diseases.items(), key=lambda x: x[1]['support_count'], reverse=True)[:3]
for i, (itemset, data) in enumerate(top_support, 1):
    print(f"  {i}. {tuple(itemset)} - {data['support_count']} patients ({data['support_pct']:.1f}%)")
    print(f"     Diseases: {', '.join(data['diseases'])}")

# Largest itemsets
largest_itemsets = sorted([(itemset, data) for itemset, data in itemset_to_diseases.items()], 
                         key=lambda x: len(x[0]), reverse=True)[:3]


DISEASES CONTAINING APRIORI ITEMSETS (2+ items)

Precomputing disease-symptom mappings...
Disease-symptom mappings computed!

Analyzing itemsets...
  Processed 10/54 itemsets...
  Processed 20/54 itemsets...
  Processed 30/54 itemsets...
  Processed 40/54 itemsets...
  Processed 50/54 itemsets...
Analysis complete!

COMPLETE RESULTS - ALL ITEMSETS (2+ SYMPTOMS):
Itemset sizes found: [2, 3, 4]

2-SYMPTOM COMBINATIONS (33 patterns)

🔹 ('nausea', 'vomiting')
   Support: 978 patients (19.9%)
Associated Diseases: (vertigo) Paroymsal  Positional Vertigo, Chronic cholestasis, Dengue, Hepatitis D, Hepatitis E, Hypoglycemia, Malaria, Typhoid, hepatitis A

🔹 ('vomiting', 'stomach_pain')
   Support: 978 patients (19.9%)
Associated Diseases: Alcoholic hepatitis, Chronic cholestasis, GERD, Hepatitis D, Hepatitis E, Jaundice, Peptic ulcer diseae, Typhoid, hepatitis A

🔹 ('fatigue', 'high_fever')
   Support: 978 patients (19.9%)
Associated Diseases: Bronchial Asthma, Chicken pox, Common Cold, Dengue,